In [2]:
import pandas as pd
import numpy as np
import sqlite3

# 1. 铸造第一题的量化并发奇盒
orders_data = {
    'user_id': ['U01', 'U01', 'U01', 'U01', 'U02'],
    'order_time': [
        '2026-06-13 10:00:00.001',
        '2026-06-13 10:00:00.002',
        '2026-06-13 10:00:00.005', # 并发奇点开始：与下一行完全相同
        '2026-06-13 10:00:00.005', # 并发奇点结束
        '2026-06-13 10:00:01.000'
    ],
    'price': [10, 20, 30, 40, 100]
}
df_orders = pd.DataFrame(orders_data).sample(frac=1).reset_index(drop=True)

# 2. 铸造第二题的断头台沙盒
logs_data = {
    'user_id': ['U01', 'U01', 'U01', 'U01', 'U01'],
    'action_time': [
        '2026-06-13 09:00:00',
        '2026-06-13 09:05:00',
        '2026-06-13 09:10:00',
        '2026-06-13 09:15:00',
        '2026-06-13 09:20:00'
    ],
    'action_type': ['click', 'purchase', 'browse', 'browse', 'click']
}
df_logs = pd.DataFrame(logs_data).sample(frac=1).reset_index(drop=True)

# 3. 物理灌入内存 SQL 擂台
conn = sqlite3.connect(':memory:')
df_orders.to_sql('orders', conn, index=False, if_exists='replace')
df_logs.to_sql('user_logs', conn, index=False, if_exists='replace')
print("=== 大厂级时空特征物理沙盒已全部死锁锁死 ===")
print(df_orders)
print(df_logs)

=== 大厂级时空特征物理沙盒已全部死锁锁死 ===
  user_id               order_time  price
0     U01  2026-06-13 10:00:00.002     20
1     U01  2026-06-13 10:00:00.001     10
2     U01  2026-06-13 10:00:00.005     40
3     U02  2026-06-13 10:00:01.000    100
4     U01  2026-06-13 10:00:00.005     30
  user_id          action_time action_type
0     U01  2026-06-13 09:20:00       click
1     U01  2026-06-13 09:05:00    purchase
2     U01  2026-06-13 09:00:00       click
3     U01  2026-06-13 09:15:00      browse
4     U01  2026-06-13 09:10:00      browse


## ⚔️ 第一题：量化特征工程——高并发“时空奇点”下的窗口坍塌

### 🏢 背景叙事

在量化交易或高频风控特征工程中，数据流极易在某些微观时空点上产生“奇点”。现有一张高频订单流表 `orders`，字段包括：`user_id`（用户ID）、`order_time`（订单时间戳，精确到毫秒）、`price`（交易价格）。

### 🎯 核心任务

请编写一段 SQL 窗口函数，在**不坍塌明细行**的前提下，计算：**每个用户在当前订单时间戳（含）之前，历史所有订单的累计交易金额（Running Total）。**

### 🚨 面试必考点与隐藏死锁（审判要害）

这道题看似极其简单，实则暗藏杀机。面试官在盯着你的以下两个底层漏洞：

1. **并发时空奇点（微观时间戳完全相同）**：如果某个恶意脚本在同一毫秒（`order_time` 完全一致）为同一个 `user_id` 涌入了 3 笔订单。当你写下 `ORDER BY order_time` 且不加任何微调时，**SQL 标准中 `RANGE`（默认）与 `ROWS` 的微观计算结果会发生恐怖的分叉。** 2. **数据重放与确定性（Deterministic）**：如果数据分布存在时间戳重复，你的特征在线上实时计算和线下离线重放时，能否保证特征矩阵的绝对对齐？
    

## ⚔️ 第二题：微观特征工程——断头台机制与“全量延迟”特征的冷启动

### 🏢 背景叙事

在电商推荐系统里，我们需要捕捉用户的“实时心跳异动”。现有一张用户行为日志表 `user_logs`，字段包括：`user_id`（用户ID）、`action_time`（行为时间线）、`action_type`（行为类型：'click', 'browse', 'purchase'）。

### 🎯 核心任务

请在不破坏明细行的前提下，同时并行提取两个高维特征：

1. **`last_purchase_time`**：当前行为发生之前（不含当前行），该用户最近一次发生 'purchase'（购买）的时间戳。
    
2. **`first_action_type`**：该用户在整个历史生命周期里，**最开始发生的第一个行为类型（Action Type）是什么**（按字母序无视时间？不，按时间正序取第一个行为）。
    

### 🚨 面试必考点与隐藏死锁（审判要害）

1. **断头台的条件过滤（Conditional Lag）**：普通的 `LAG` 只能盲目拿上一行。但这道题要求拿“最近一次 **'purchase'** 的行”。在当前行和上一次购买之间，可能隔着 100 行 'click'。你如何利用窗口取值函数，精准刺穿这 100 行噪音，斩断过去，拿到那次购买的时间？
    
2. **绝对取值的隐式死锁**：你在提取 `first_action_type` 时，是否会因为 `ORDER BY` 的注入而导致窗口退化？你如何处理那些新注册、没有任何历史行为的用户的冷启动 `NULL` 值？

In [3]:
# 第一题
# SQL轨道
sql_query = """
SELECT  user_id,
        order_time,
        SUM(price) OVER(PARTITION BY user_id ORDER BY order_time ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM orders
ORDER BY user_id,order_time ASC;
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

  user_id               order_time  running_total
0     U01  2026-06-13 10:00:00.001             10
1     U01  2026-06-13 10:00:00.002             30
2     U01  2026-06-13 10:00:00.005             70
3     U01  2026-06-13 10:00:00.005            100
4     U02  2026-06-13 10:00:01.000            100


In [ ]:
# PANDAS轨道
df_order_sorted = df_orders.sort_values(by=['user_id','order_time'])
df_order_sorted['running_total'] = (
    df_order_sorted
    .groupby('user_id')['price']
    .cumsum()
    )
print(df_order_sorted)

  user_id               order_time  price  running_total
1     U01  2026-06-13 10:00:00.001     10             10
0     U01  2026-06-13 10:00:00.002     20             30
2     U01  2026-06-13 10:00:00.005     40             70
4     U01  2026-06-13 10:00:00.005     30            100
3     U02  2026-06-13 10:00:01.000    100            100


In [21]:
# 第二题
# SQL轨道
sql_query = """
SELECT 
    user_id,
    action_time,
    action_type,
    
    -- 1. 跨越点击噪声，精准斩断过去，抓取最近一次购买时间
    LAST_VALUE(CASE WHEN action_type = 'purchase' THEN action_time END) OVER(
        PARTITION BY user_id
        ORDER BY action_time ASC
        -- 核心防御线：终点卡在 1 PRECEDING,严格排除当前行本身,严防特征自污染
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS last_purchase_time,
    
    -- 2. 锚定生命周期起点，锁定用户的冷启动初始行为
    FIRST_VALUE(action_type) OVER(
        PARTITION BY user_id 
        ORDER BY action_time ASC -- 顺着时间轴看
        -- 显式死锁全量窗口：让视线看穿过去和未来，牢牢扣住开天辟地的第一行
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS first_action_type

FROM user_logs
-- 特征矩阵交付前，定海神针最后一次拉直全局输出流
ORDER BY user_id, action_time ASC;
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

  user_id          action_time action_type   last_purchase_time  \
0     U01  2026-06-13 09:00:00       click                 None   
1     U01  2026-06-13 09:05:00    purchase                 None   
2     U01  2026-06-13 09:10:00      browse  2026-06-13 09:05:00   
3     U01  2026-06-13 09:15:00      browse                 None   
4     U01  2026-06-13 09:20:00       click                 None   

  first_action_type  
0             click  
1             click  
2             click  
3             click  
4             click  


In [42]:
import pandas as pd

# 1. 核心铁轨防御：交付计算前，必须让时间绝对正序排列
df_logs_sorted = df_logs.sort_values(by=['user_id', 'action_time'], ascending=[True, True]).copy()

# ==========================================
# 🚀 PANDAS 双轨并行特征提取矩阵
# ==========================================

# 轨道一：上一次购买时间 (对应 LAST_VALUE + 1 PRECEDING)
purchase_mask = df_logs_sorted['action_type'] == 'purchase'
df_logs_sorted['only_purchase_time'] = df_logs_sorted['action_time'].where(purchase_mask)
# shift(1) 完美实现 1 PRECEDING 的物理挡光板
df_logs_sorted['shifted_purchase_time'] = df_logs_sorted.groupby('user_id')['only_purchase_time'].shift(1)
df_logs_sorted['last_purchase_time'] = df_logs_sorted.groupby('user_id')['shifted_purchase_time'].ffill().fillna('1970-01-01 00:00:00')

# 轨道二：生命周期初始行为 (对应 FIRST_VALUE + UNBOUNDED PRECEDING AND FOLLOWING)
# .transform('first') 自动在分组高墙内抓取最上方的一行，并像广播一样回填所有明细行
df_logs_sorted['first_action_type'] = df_logs_sorted.groupby('user_id')['action_type'].transform('first')

# 2. 物理大扫除：剔除过程中的辅助虚无列
df_final_matrix = df_logs_sorted.drop(columns=['only_purchase_time', 'shifted_purchase_time'])

print("=== 🪐 推荐系统时空特征矩阵对账成功 🪐 ===")
print(df_final_matrix)

=== 🪐 推荐系统时空特征矩阵对账成功 🪐 ===
  user_id          action_time action_type   last_purchase_time  \
2     U01  2026-06-13 09:00:00       click  1970-01-01 00:00:00   
1     U01  2026-06-13 09:05:00    purchase  1970-01-01 00:00:00   
4     U01  2026-06-13 09:10:00      browse  2026-06-13 09:05:00   
3     U01  2026-06-13 09:15:00      browse  2026-06-13 09:05:00   
0     U01  2026-06-13 09:20:00       click  2026-06-13 09:05:00   

  first_action_type  
2             click  
1             click  
4             click  
3             click  
0             click  
